# 06b · Extract all directions (HF) — probe + steering vectors, per model

**Stage 2 of 2.** Reads activations off a fixed frame and writes, for **every** model at **mid→late
layers**, three kinds of direction:

1. **Desirability probe** — regression of task-prompt activations → Thurstonian μ (à la `04`). Reads
   the task text; **generates nothing**. Saved for *every* band layer + the standardisation stats
   (`mean`/`scale`) needed to apply model A's probe to model B (transfer matrix).
2. **Desirability steering vector** — repeng contrast, top-K vs bottom-K tasks by μ (à la `05`). Also
   a task-prompt read, no generation.
3. **Pathology vectors** (clinical 10 + PC 7) — repeng on **this model's own** persona completions
   from `06a` (`generations_{set}_{model}.jsonl`).

**Frozen frame:** the task-text stimulus set (intersection of task_ids measured across all models);
only the labels (μ) are per-model. **Layer alignment:** paper-repo block-`L` output == repeng
`hidden_states[L+1]`, so probe@L and vector@L are the same residual point. Run once, after every
organism has a μ run (`02`) **and** its `06a` generations. Output → `DRIVE/directions_v1/`.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
import sys, subprocess, pathlib
PC = pathlib.Path("/content/Predictive_coding")
if not PC.exists():
    subprocess.check_call(["git","clone","https://github.com/ChuloIva/Predictive_coding.git", str(PC)])
LAB = PC / "steering_lab"
for p in (str(LAB), str(LAB/"third_party"/"repeng")):
    if p not in sys.path: sys.path.insert(0, p)
print("steering + repeng on path:", (LAB/"steering"/"extract.py").exists())

In [ ]:
# HF-only (no vLLM here). repeng from git (PyPI pins numpy<2).
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece
%pip install -q -U git+https://github.com/vgel/repeng.git
import importlib
for _m in ("numpy","scipy","sklearn","transformers","repeng"):
    try: importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))
    except Exception as _e: print(_m, "FAILED:", type(_e).__name__, str(_e)[:160])

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set")

In [ ]:
DRIVE = mount_drive()
use_probe_repo()               # `import src.*` == paper repo (models/task_data/measurement)
import pathlib
assert DRIVE is not None
OUT = DRIVE / "directions_v1"; OUT.mkdir(parents=True, exist_ok=True)
print("directions ->", OUT)

## 2. Config
Same `MODELS` as `06a`. Layer band = fractional depth → absolute blocks (Qwen3-8B = 36).

In [ ]:
import json
MODELS = json.load(open("/content/dt_rl/notebooks/organisms.json"))["models"]  # edit HF ids THERE
print("organisms:", ", ".join(f"{m['name']}{'' if m['hf'] else '(-)'}" for m in MODELS))

LAYER_BAND   = (0.45, 0.95)  # fractional -> mid..late blocks
SELECTOR     = "task_mean"   # probe pooling (matches 04); or "task_last"
PROBE_BATCH  = 8
PROBE_MAXTOK = 1024
DESIRE_TOPK  = 300
EXTRACT_BATCH = 16
print(f"{sum(1 for m in MODELS if m['hf'])}/{len(MODELS)} checkpoints; band={LAYER_BAND}")

## 3. Helpers — μ + task text, probe activations, probe fit (mirror `04`)

In [ ]:
import json, gc, csv, numpy as np, torch
from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from src.models.huggingface_model import HuggingFaceModel
from src.measurement.storage.loading import load_run_utilities
from src.task_data.loader import load_filtered_tasks, FILE_MAPPING

def find_run_dir(exp_id):
    roots = [pathlib.Path("results/experiments")/exp_id]
    if DRIVE: roots.append(DRIVE/"measurements"/exp_id)
    for root in roots:
        hits = list(root.glob("**/thurstonian_*.csv")) if root.exists() else []
        if hits: return hits[0].parent
    return None

def load_mu(exp_id):
    rd = find_run_dir(exp_id)
    if rd is None: return None
    mu, ids = load_run_utilities(rd)
    return dict(zip(ids, [float(m) for m in mu]))

def build_stimuli(tok, ids, text_by_id):
    out = []
    for tid in ids:
        p = text_by_id[tid]; t = tok(p, add_special_tokens=False).input_ids
        if len(t) > PROBE_MAXTOK: p = tok.decode(t[:PROBE_MAXTOK])
        out.append([{"role":"user","content":p}])
    return out

@torch.inference_mode()
def extract_X(model, stimuli, layers, selector):
    buf = {L: [] for L in layers}
    for i in tqdm(range(0, len(stimuli), PROBE_BATCH), desc="activations", unit="batch"):
        res = model.get_activations_batch(stimuli[i:i+PROBE_BATCH], layers, [selector])
        for L in layers: buf[L].append(res[selector][L])
    return {L: np.concatenate(buf[L], 0) for L in layers}

def _pairwise_acc(pred, true):
    dp = np.sign(pred[:,None]-pred[None,:]); dt = np.sign(true[:,None]-true[None,:])
    m = np.triu(np.ones_like(dp, bool), 1); return float((dp[m]==dt[m]).mean())

def fit_probe(X, mu, seed=0, test_frac=0.2, alphas=np.logspace(1,5,25)):
    # Ridge probe: standardized activations -> mu; direction returned in RAW activation space.
    n = len(mu); idx = np.random.default_rng(seed).permutation(n); nte = int(n*test_frac)
    te, tr = idx[:nte], idx[nte:]
    sc = StandardScaler().fit(X[tr]); Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
    best = None
    for al in alphas:
        m = Ridge(alpha=al).fit(Xtr, mu[tr]); r = pearsonr(m.predict(Xte), mu[te])[0]
        if best is None or r > best["r"]: best = {"r":r,"alpha":al,"m":m}
    m = best["m"]; pred_te = m.predict(Xte)
    w_std, b_std = m.coef_, float(m.intercept_)
    w_raw = w_std / sc.scale_; b_raw = b_std - float(np.sum(w_std*sc.mean_/sc.scale_))
    nrm = np.linalg.norm(w_raw) or 1.0
    return {"r":float(best["r"]), "rho":float(spearmanr(pred_te, mu[te])[0]),
            "pair_acc":_pairwise_acc(pred_te, mu[te]), "alpha":float(best["alpha"]),
            "w_raw":w_raw.astype(np.float32), "b_raw":b_raw, "unit":(w_raw/nrm).astype(np.float32),
            "mean":sc.mean_.astype(np.float32), "scale":sc.scale_.astype(np.float32),
            "scores":(X @ w_raw + b_raw).astype(np.float32)}
print("helpers ready")

## 4. Freeze the stimulus frame + resolve layer band

In [ ]:
N_BLOCKS = 36
lo, hi = LAYER_BAND
LAYERS_ABS = list(range(int(lo*N_BLOCKS), int(hi*N_BLOCKS)+1))
print(f"band {LAYER_BAND} -> blocks {LAYERS_ABS[0]}..{LAYERS_ABS[-1]} ({len(LAYERS_ABS)} layers)")

MU_BY_EXP = {}
for spec in MODELS:
    mu = load_mu(spec["exp_id"])
    if mu is not None:
        MU_BY_EXP[spec["exp_id"]] = mu; print(f"[mu] {spec['name']:10s} {len(mu)} tasks")
    else:
        print(f"[mu] {spec['name']:10s} -- no run (probe+desirability skip)")
assert MU_BY_EXP, "no mu runs — run 02 for base first"

common = set.intersection(*[set(m) for m in MU_BY_EXP.values()])
_txt = {t.id: t.prompt for t in load_filtered_tasks(n=10**9, origins=list(FILE_MAPPING), task_ids=common)}
FROZEN_IDS = sorted(i for i in common if i in _txt)
TEXT_BY_ID = {i: _txt[i] for i in FROZEN_IDS}
json.dump(FROZEN_IDS, open(OUT/"frozen_task_ids.json","w"))
print(f"FROZEN task set: {len(FROZEN_IDS)} tasks (intersection over {len(MU_BY_EXP)} measured models)")

## 5. Main loop — per model: probe + desirability vector + pathology vectors
One model in memory at a time. Probe uses paper-repo `HuggingFaceModel`; steering uses `steer`+repeng.
Pathology vectors read **this model's own** `06a` generations from Drive.

In [ ]:
from steering import config as scfg, extract, steer
from steering.personas import PERSONA_BY_ID
from steering.personas_pc import PC_PERSONA_BY_ID
from repeng import DatasetEntry
CAA_SETS = [("clinical", PERSONA_BY_ID), ("pc", PC_PERSONA_BY_ID)]

def _save_probe(name, fits, ids, mu_aligned):
    L0 = LAYERS_ABS
    np.savez(OUT/f"probe_{name}_all.npz", layers=np.array(L0),
             w_raw=np.stack([fits[L]["w_raw"] for L in L0]),
             b_raw=np.array([fits[L]["b_raw"] for L in L0], dtype=np.float32),
             unit=np.stack([fits[L]["unit"] for L in L0]),
             mean=np.stack([fits[L]["mean"] for L in L0]),
             scale=np.stack([fits[L]["scale"] for L in L0]),
             r=np.array([fits[L]["r"] for L in L0], dtype=np.float32),
             rho=np.array([fits[L]["rho"] for L in L0], dtype=np.float32),
             pair_acc=np.array([fits[L]["pair_acc"] for L in L0], dtype=np.float32))
    bestL = max(L0, key=lambda L: fits[L]["r"])
    json.dump({"model":name,"selector":SELECTOR,"layers":L0,"best_layer":int(bestL),"n_tasks":len(ids),
               "d_model":int(len(fits[bestL]["w_raw"])),
               "heldout_pearson":{int(L):fits[L]["r"] for L in L0},
               "heldout_spearman":{int(L):fits[L]["rho"] for L in L0},
               "pairwise_acc":{int(L):fits[L]["pair_acc"] for L in L0}},
              open(OUT/f"probe_{name}_meta.json","w"), indent=2)
    with open(OUT/f"scores_{name}_L{bestL}.csv","w",newline="") as f:
        w = csv.writer(f); w.writerow(["task_id","mu","probe_score"])
        for tid, s in zip(ids, fits[bestL]["scores"]): w.writerow([tid, float(mu_aligned[tid]), float(s)])
    return bestL

def _desirability_entries(tok, mu_by):
    kept = [t for t in FROZEN_IDS if t in mu_by]
    kmu = np.array([mu_by[t] for t in kept]); order = np.argsort(kmu)
    K = min(DESIRE_TOPK, len(kept)//4)
    hi_ids = [kept[i] for i in order[-K:]]; lo_ids = [kept[i] for i in order[:K]]
    def s(text):
        full = tok.apply_chat_template([{"role":"user","content":text}], tokenize=False,
                                       add_generation_prompt=False)
        j = full.rfind(text.strip()); return full[:j+len(text.strip())] if j!=-1 else text
    ents = [DatasetEntry(positive=s(TEXT_BY_ID[h]), negative=s(TEXT_BY_ID[l]))
            for h,l in zip(hi_ids, lo_ids)]
    return ents, K, float(kmu[order[-K]]), float(kmu[order[K-1]])

MANIFEST = {"layer_band":LAYER_BAND, "layers_abs":LAYERS_ABS, "selector":SELECTOR,
            "frozen_task_ids":len(FROZEN_IDS), "models":{}}

for spec in MODELS:
    name, hf, exp = spec["name"], spec["hf"], spec["exp_id"]
    if not hf: print(f"\n#### skip {name}: no checkpoint ####"); continue
    has_mu = exp in MU_BY_EXP; mu_aligned = MU_BY_EXP.get(exp, {})
    print(f"\n{'='*64}\n{name} :: {hf}  (mu={'yes' if has_mu else 'NO'})\n{'='*64}")
    rec = {"hf":hf, "has_mu":has_mu, "outputs":[]}

    # PHASE A: probe (task-prompt activations -> mu)
    if has_mu:
        mu_vec = np.array([mu_aligned[t] for t in FROZEN_IDS], dtype=np.float64)
        model = HuggingFaceModel(hf, dtype="bfloat16", device="cuda")
        X = extract_X(model, build_stimuli(model.tokenizer, FROZEN_IDS, TEXT_BY_ID), LAYERS_ABS, SELECTOR)
        fits = {L: fit_probe(X[L], mu_vec) for L in LAYERS_ABS}
        bestL = _save_probe(name, fits, FROZEN_IDS, mu_aligned)
        # cache the mean-pooled task activations [n, nL, d] fp16 -> 07 does transfer + CKA offline
        np.savez(OUT/f"acts_{name}.npz", task_ids=np.array(FROZEN_IDS), layers=np.array(LAYERS_ABS),
                 mu=mu_vec.astype(np.float32),
                 X=np.stack([X[L] for L in LAYERS_ABS], axis=1).astype(np.float16))
        print(f"[{name}] probe bestL={bestL} r={fits[bestL]['r']:.3f} "
              f"(band {min(f['r'] for f in fits.values()):.3f}..{max(f['r'] for f in fits.values()):.3f})")
        rec["outputs"] += [f"probe_{name}_all.npz", f"acts_{name}.npz"]; rec["best_probe_layer"] = int(bestL)
        del model, X; gc.collect(); torch.cuda.empty_cache()
    else:
        print(f"[{name}] no mu -> skip probe + desirability")

    # PHASE B: steering vectors (repeng)
    model, tok = steer.load_model_and_tokenizer(hf, dtype="bfloat16", device_map="cuda",
                                                hf_token=os.environ.get("HF_TOKEN") or None)
    model.eval()
    ecfg = scfg.ExtractConfig(model_name=hf); ecfg.batch_size = EXTRACT_BATCH; ecfg.hidden_layers = LAYERS_ABS

    if has_mu:
        ents, K, mu_hi, mu_lo = _desirability_entries(tok, mu_aligned)
        print(f"[{name}] desirability: {K} hi/lo pairs (mu hi>={mu_hi:.2f} vs lo<={mu_lo:.2f})")
        desir = extract.extract_vectors(model, tok, {"desirability":ents}, ecfg)
        p = OUT/f"control_vectors_desirability_{name}.pkl"
        extract.save_bundle(desir, str(p), model_name=hf, cfg=ecfg, pairs={"desirability":ents},
                            meta_path=str(p.with_name(p.stem+"_meta.json"))); rec["outputs"].append(p.name)

    for set_name, pbid in CAA_SETS:
        gpath = OUT/f"generations_{set_name}_{name}.jsonl"
        if not gpath.exists():
            print(f"[{name}/{set_name}] MISSING {gpath.name} -> run 06a first; skipping"); continue
        records = extract.load_records(str(gpath))
        pairs = extract.build_pairs(records, tok, ecfg, persona_by_id=pbid)
        vecs = extract.extract_vectors(model, tok, pairs, ecfg)
        p = OUT/f"control_vectors_{set_name}_{name}.pkl"
        extract.save_bundle(vecs, str(p), model_name=hf, cfg=ecfg, pairs=pairs,
                            meta_path=str(p.with_name(p.stem+"_meta.json"))); rec["outputs"].append(p.name)
        print(f"[{name}/{set_name}] {len(vecs)} vectors x {len(LAYERS_ABS)} layers")

    MANIFEST["models"][name] = rec
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"[{name}] done. GPU:", round(torch.cuda.memory_allocated()/1e9,2), "GB")

json.dump(MANIFEST, open(OUT/"manifest.json","w"), indent=2)
print("\nMANIFEST ->", OUT/"manifest.json")

## 6. Sanity — cross-model probe cosine @ mid layer

In [ ]:
import numpy as np, pickle
def _unit(v): v=np.asarray(v,np.float32); n=np.linalg.norm(v); return v/n if n else v
Lmid = LAYERS_ABS[len(LAYERS_ABS)//2]
probes = {}
for spec in MODELS:
    f = OUT/f"probe_{spec['name']}_all.npz"
    if f.exists():
        z = np.load(f); probes[spec["name"]] = z["unit"][list(z["layers"]).index(Lmid)]
names = list(probes)
if len(names) >= 2:
    print(f"desirability PROBE cosine across models @ L{Lmid}")
    print("           " + "  ".join(f"{n:>9s}" for n in names))
    for aa in names:
        print(f"{aa:>9s}  " + "  ".join(f"{_unit(probes[aa])@_unit(probes[bb]):+.3f}".rjust(9) for bb in names))
else:
    print("need >=2 probes; have:", names)

## Outputs (`DRIVE/directions_v1/`)
Per model `<m>`: `probe_<m>_all.npz` (+ `_meta.json`, `scores_<m>_L<best>.csv`),
`control_vectors_desirability_<m>.pkl`, `control_vectors_clinical_<m>.pkl`,
`control_vectors_pc_<m>.pkl` (+ each `_meta.json`). Shared: `frozen_task_ids.json`, `manifest.json`
(and the `generations_*_<m>.jsonl` from `06a`). All directions are 4096-d, comparable across models
(shared Qwen3-8B basis) and across estimators at equal layer id. Next: `07` — organism cosine matrix,
probe transfer matrix, per-layer CKA.